# Morning class 27/08 — Worksheet 09 SOLUTIONS: defining functions   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Question 2 is the one to read even if you got it right. It is the difference
between a function that works and one that only looks like it works.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 09 — Defining functions. Run this once.
PI = 3.14159265359

orders_2019 = [2.50, 4.50, 2.00, 2.75, 2.50, 5.00, 8.00, 2.25]
orders_2020 = [3.50, 7.50, 1.10, 1.75, 3.50, 6.30, 4.60, 5.25]

readings = [4.2, 0.0, 7.9, 3.1, 12.5]

# For the login question -- what somebody typed, instead of input().
attempts = ["hunter2", "password", "D4t4Sc13nc3!"]
bad_attempts = ["aaa", "bbb", "ccc", "D4t4Sc13nc3!"]

print(len(orders_2019), "orders in 2019,", len(orders_2020), "in 2020")

PART A — def, call, return

### Question 1

The circle function from slide 14. -> `31.4159265359`, `6.28318530718`, `37.69911184308`.

The body ran three times from one definition. That is the entire point of a
function, and it is why the last line can call it twice in one expression
without repeating a line of arithmetic.

`circumference` inside the body is a **local** variable — it is created when
the call starts and gone when it ends. Try to print it after the call and
you get a `NameError`; worksheet 11 is about why.

`PI` was **not** passed in. The function reached out to the surrounding
code to find it, which works and is worth being suspicious of — worksheet
11 Q2 is where that stops being convenient.

In [ ]:
def circumference_of_circle(radius):
    circumference = 2 * PI * radius
    return circumference

print(circumference_of_circle(5))
print(circumference_of_circle(1))
print(circumference_of_circle(5) + circumference_of_circle(1))

### Question 2

Print versus return. -> `ANA!`, `ANA!`, then **`a is None`** and `b is ANA!`.

Four calls, three lines of output before the last two prints, and one of
them came from nowhere useful.

- `shout_print("ana")` printed `ANA!` as a **side effect** and evaluated to
  `None`.
- `shout_return("ana")` produced `ANA!` and nothing caught it, so it was
  computed and thrown away — no output at all from that line.

So `a is None`. **A function with no `return` returns `None`**, always,
silently. It looked like it worked because the printing happened to be
visible in a notebook — and the moment you try to use the result, add it to
something, or write it to a file, you have `None`.

The rule: a function that **computes** something should `return` it. Print
at the top level, where you can see what is going on. A function that both
computes and prints is a function you cannot test and cannot reuse.

In [ ]:
def shout_print(name):
    print(name.upper() + "!")

def shout_return(name):
    return name.upper() + "!"

shout_print("ana")            # prints as a side effect
shout_return("ana")           # returns -- and nothing catches it, so nothing shows

a = shout_print("ana")
b = shout_return("ana")

print("a is", a)
print("b is", b)

### Question 3

Returning two values. -> `0.0 12.5`, then `(0.0, 12.5)`, then `<class 'tuple'>`.

`return smallest, largest` does not return two things — it returns **one
tuple** holding two things. The commas build a tuple, exactly as they did
in the 26/08 session; the parentheses in the printed output are Python's,
not yours.

`low, high = min_max(readings)` then unpacks it, which is the same
unpacking as `for name, course in enrollment`. It will raise `ValueError`
the day the function returns three things and a caller still expects two.

Starting both variables at `values[0]` rather than at `0` matters: start
`smallest` at `0` and this function can never report a negative minimum.
It would also be wrong here — `0.0` is genuinely in `readings`, but only by
luck.

In [ ]:
def min_max(values):
    smallest = values[0]
    largest = values[0]
    for v in values:
        if v < smallest:
            smallest = v
        if v > largest:
            largest = v
    return smallest, largest

low, high = min_max(readings)
print(low, high)

print(min_max(readings))
print(type(min_max(readings)))

### Question 4

A docstring. -> the two-line description, then `help` printing `min_max(values)` with the same text indented under it.

A docstring is a plain string as the **first statement** of the body. It is
not a comment: Python keeps it, attaches it to the function as `__doc__`,
and `help()` reads it back. A `#` comment is discarded at parse time and
cannot be recovered.

So the audience is different. Comments explain the implementation to
someone reading the code; the docstring explains the **contract** to
someone calling it who will never open the file.

Note what the docstring says beyond the obvious: *assumes `values` is not
empty*. That is the part worth writing down — the conditions under which
the function is allowed to be trusted. Q10 is the same function with that
assumption removed.

In [ ]:
def min_max(values):
    """Return the smallest and largest value in `values`, as a tuple.

    Assumes `values` is not empty.
    """
    smallest = values[0]
    largest = values[0]
    for v in values:
        if v < smallest:
            smallest = v
        if v > largest:
            largest = v
    return smallest, largest

print(min_max.__doc__)
help(min_max)

### Question 5

The function as a value. -> `<function circumference_of_circle at 0x…>`, then `31.4159265359` twice, then `True`.

Without parentheses you get the function **object** — a value, with a type
and an address, that happens to be callable. With parentheses you get the
result of running it. (The hex address will differ on your machine and
means nothing.)

That first line is what you see when you forget the parentheses, and it is
worth recognising: `print(my_func)` and `if my_func:` both "work", and a
function object is always truthy, so `if get_flag:` is `True` whatever
`get_flag()` would have returned.

`circ = circumference_of_circle` copies the name, not the function, so
`circ is circumference_of_circle` is `True` — two labels on one object.
Being able to pass a function around like this is what makes `map`,
`filter` and `sorted(key=...)` possible; worksheet 12 uses it.

In [ ]:
print(circumference_of_circle)
print(circumference_of_circle(5))

circ = circumference_of_circle       # no parentheses -- copying the name, not calling
print(circ(5))
print(circ is circumference_of_circle)

PART B — Returning early, and returning nothing

### Question 6

Three returns and unreachable code. -> `negative`, `zero`, `positive`. **`still running` never appeared.**

`return` leaves the function **immediately** — like `break`, but for the
whole function rather than one loop. Every call matched one of the three
conditions, so the last line was never reached on any of them.

Early returns like this are usually clearer than one nested `if`/`elif`
chain assigning to a variable, because each branch is finished the moment
it is decided and nothing below can change it.

But note the risk, which is the same as worksheet 01 Q5: if the three
conditions did not cover every input, the function would fall off the
bottom and return `None` — and `None` would flow into the caller with no
error. Here `< 0`, `== 0` and `> 0` are exhaustive for numbers. Check that
they are, or make the last branch an unconditional `return`.

In [ ]:
def classify(n):
    if n < 0:
        return "negative"
    if n == 0:
        return "zero"
    if n > 0:
        return "positive"
    print("still running")

print(classify(-3))
print(classify(0))
print(classify(7))

# Never. `return` leaves the function immediately -- like `break`, but for the
# whole function, not just a loop. Anything below the return that fires is
# unreachable for that call.

### Question 7

Slide 17's fraud rules, all four branches. -> `1.0 large, CA`; `0.5 large, not CA`; `1.0 small, many countries`; `0.25 small, few countries`.

Four calls to cover a function with two nested decisions — that is 2 × 2,
the same multiplication as nested loops. Add one more condition and you
need eight.

Worth noticing: `1.00` printed as `1.0` and `0.50` as `0.5`. The trailing
zeros were never stored; they were formatting in the source. If this score
is going into a report, format it at the point of display with
`f"{score:.2f}"`.

The deck's own call is the third one, `predict_fraud(1.50, 'CA', 3)`, and
slide 17 cuts off before showing its answer. It is `1.0` — a £1.50
transaction scored as maximum fraud risk because the card has been used in
three countries.

In [ ]:
def predict_fraud(amount_usd, card_country, countries_used):
    if amount_usd > 20.00:
        if card_country == "CA":
            fraud_score = 1.00
        else:
            fraud_score = 0.50
    else:
        if countries_used > 2:
            fraud_score = 1.00
        else:
            fraud_score = 0.25
    return fraud_score

print(predict_fraud(50.00, "CA", 1), "large, CA")
print(predict_fraud(50.00, "US", 1), "large, not CA")
print(predict_fraud(1.50, "CA", 3), "small, many countries")
print(predict_fraud(1.50, "CA", 1), "small, few countries")

PART C — Functions that use other functions

### Question 8

Functions calling functions. -> 2019: `{'name': '2019', 'count': 8, 'total': 29.5, 'average': 3.6875, 'min': 2.0, 'max': 8.0}`; 2020: `{'name': '2020', 'count': 8, 'total': 33.5, 'average': 4.1875, 'min': 1.1, 'max': 7.5}`.

`describe` does not know how to average or how to find a minimum. It calls
two functions that do, and spends its own lines on the shape of the answer.
That layering is what "reusability" actually looks like day to day — small
functions that do one thing, and one function that arranges them.

Returning a **dictionary** rather than six loose values means the caller
reads `result["average"]` instead of `result[3]`, and adding a seventh
figure next month breaks nobody's unpacking.

The two calls are the whole argument for functions. Slides 44 and 48 show
this calculation written out twice, once per year, with `sum_2019`/`cnt_2019`
and `sum_2020`/`cnt_2020`. Two years is annoying; ten is a bug farm,
because the tenth copy is where the typo lives.

In [ ]:
def average(values):
    return sum(values) / len(values)

def describe(name, values):
    low, high = min_max(values)
    return {
        "name": name,
        "count": len(values),
        "total": sum(values),
        "average": average(values),
        "min": low,
        "max": high,
    }

print(describe("2019", orders_2019))
print(describe("2020", orders_2020))

### Question 9

Slide 16's login, returning instead of printing. -> three attempt lines then `attempts -> True`; three attempt lines then `bad_attempts -> False`.

Both runs used all three tries — the correct password is the third guess in
`attempts` and the fourth in `bad_attempts`, which the cap never reaches.

The important difference from the deck is the **return**. Slide 16's
`login()` prints `Login Successful!` and returns `None`, so the calling
code cannot tell whether the login worked; it would have to scrape stdout.
Returning `True`/`False` means the caller can branch on it, test it, and log
it.

That is the general shape of turning a script into a function: the printing
is for a human watching, the return value is for the program. Keep them
separate, and put the printing at the outermost layer.

(The deck also uses `input()` here, which is why its version can never be
re-run in a notebook. Passing the guesses in as a list makes the function
testable — you can call it a hundred times without typing anything.)

In [ ]:
def login(guesses, password):
    tries = 0
    for guess in guesses:
        if tries >= 3:
            break
        tries = tries + 1
        print("attempt", tries)
        if guess == password:
            return True
    return False

print("attempts    ->", login(attempts, "D4t4Sc13nc3!"))
print("bad_attempts ->", login(bad_attempts, "D4t4Sc13nc3!"))

### Question 10

The empty case. -> `5.54`, then `None`.

`if not values:` is worksheet 01's truthiness doing a real job — an empty
list is falsy, so this catches `[]` without a `len()` call.

And 5.54 is the mean of five readings **including the 0.0**. Whether that
zero is a real measurement or a missing one changes the answer to 6.925,
and nothing in the data says which. That is extra practice 05 Q3 again.

On returning `None`: it is honest — there is no average of nothing — but it
moves the problem to the caller, and `None > 5` raises `TypeError` while
`None` printed in a report just says `None`. The alternatives are to raise
deliberately, or to return `0.0` and lie. There is no free option; pick one
deliberately and write it in the docstring.

In [ ]:
def safe_average(values):
    if not values:
        return None
    return sum(values) / len(values)

print(safe_average(readings))
print(safe_average([]))

# average([]) from Q8 would raise ZeroDivisionError, because len([]) is 0.
# Returning None says "there is no answer" -- but the CALLER now has to check
# for it, and `None > 5` raises. There is no free option here; the choice is
# between raising and making every caller handle a None.

### Question 11

Too few arguments. -> `1.0`, then `TypeError: predict_fraud() missing 1 required positional argument: 'countries_used'`.

Python checks the call **before** running a single line of the body, and
names the parameter you left out. Too many arguments fails the same way
(`takes 3 positional arguments but 4 were given`).

This is one of the few places in the whole course where getting it wrong
fails loudly and precisely. Compare it with worksheet 02's `zip` silently
dropping a city, or worksheet 08's duplicate product ids: the language
protects the function signature far more carefully than it protects your
data.

Worksheet 10 is about the ways to relax this — defaults, keywords, `*args`
— and about what you give up each time you do.

In [ ]:
print(predict_fraud(50.00, "CA", 1))

# This is SUPPOSED to raise: TypeError: predict_fraud() missing 1 required
# positional argument: 'countries_used'.
#
# Too FEW arguments and too MANY both raise TypeError, immediately, before a
# single line of the body runs -- Python checks the call signature first.
# That is worth appreciating: unlike almost everything else in this course,
# getting the arguments wrong fails loudly rather than quietly.
print(predict_fraud(50.00, "CA"))